# Day 13 — Tools as functions (a tool registry)

**Phase 2 · From Generative to Agentic · ~35 min · No setup**

## 🎯 Objective
Understand that an agent's actions are just **functions** (tools), and build the
**registry + dispatcher** every framework uses under the hood.

## 🧠 Concept
An agent affects the world through **tools**: functions with a **name**,
**description**, and **arguments**. Before letting an LLM *choose* a tool, build the
plumbing yourself.

```mermaid
flowchart LR
    D["Decision: use <b>add</b> a=2 b=40"] --> R{"Registry<br/>lookup"}
    R -- found --> C["add(2, 40)"] --> O["42"]
    R -- missing --> E["Error: unknown tool"]
```

A registry lets you add a tool by writing one decorated function — exactly how
`@tool` decorators work in LangChain, Semantic Kernel, and AutoGen.

## 🔬 Exercise
Implement `register(name, description)` (a decorator) and `dispatch(name, **kwargs)`.

## ✅ Done when
- Adding a tool is one decorated function; unknown tools fail gracefully.

## 📝 Reflect
1. Why does a good *description* matter once an LLM is choosing the tool?
2. What's risky about passing model-provided `**kwargs` straight into a function?

## 🔭 Tomorrow
Day 14: connect the brain — an LLM chooses tools in a loop (your first real agent).


## 🔬 Your turn
Complete the `TODO`s below, then run the cell (**Shift+Enter**).

In [ ]:
from dataclasses import dataclass
from typing import Callable


@dataclass
class Tool:
    name: str
    description: str
    func: Callable[..., str]


REGISTRY: dict[str, Tool] = {}


def register(name, description):
    """Decorator that stores the function in REGISTRY as a Tool.

    TODO: inside deco, set REGISTRY[name] = Tool(name, description, func); return func.
    """
    def deco(func):
        # TODO
        return func
    return deco


def dispatch(name, **kwargs):
    """Look up a tool by name and call it.

    TODO: if name not in REGISTRY -> return f"Error: unknown tool '{name}'";
          else return REGISTRY[name].func(**kwargs)
    """
    raise NotImplementedError


@register("add", "Add two integers a and b")
def add(a, b):
    return str(a + b)


@register("shout", "Return the message in UPPERCASE")
def shout(text):
    return text.upper()


if __name__ == "__main__":
    print("tools:", list(REGISTRY))
    print(dispatch("add", a=2, b=40))
    print(dispatch("shout", text="agents are fun"))
    print(dispatch("teleport"))


---
---
## 🔒 Solution
*Try it yourself first!* Run the cell below only when you're ready to compare.

In [ ]:
from dataclasses import dataclass
from typing import Callable


@dataclass
class Tool:
    name: str
    description: str
    func: Callable[..., str]


REGISTRY: dict[str, Tool] = {}


def register(name, description):
    def deco(func):
        REGISTRY[name] = Tool(name=name, description=description, func=func)
        return func
    return deco


def dispatch(name, **kwargs):
    tool = REGISTRY.get(name)
    if tool is None:
        return f"Error: unknown tool '{name}'"
    try:
        return tool.func(**kwargs)
    except TypeError as exc:
        return f"Error: bad arguments for '{name}': {exc}"


@register("add", "Add two integers a and b")
def add(a, b):
    return str(a + b)


@register("shout", "Return the message in UPPERCASE")
def shout(text):
    return text.upper()


if __name__ == "__main__":
    print("tools:", list(REGISTRY))
    print(dispatch("add", a=2, b=40))
    print(dispatch("shout", text="agents are fun"))
    print(dispatch("teleport"))
